# Investment Lab — Hypothesis Lab
This notebook is the main place to run and inspect hypotheses. The real research logic lives in `src/investment_lab/research/runner.py`; this notebook only loads a spec, runs it, and shows the outputs.

**Current test:** if a stock rises by more than 3% for three consecutive trading days, is the following trading-day return greater than 1%?


In [1]:
import sys
print(sys.executable)

/Users/borisyakovlev/Documents/Jane/Current Version/investment_lab_poc/.venv/bin/python


In [2]:
from pathlib import Path
import json, sys

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from investment_lab.research.runner import load_hypothesis, run_hypothesis, format_result

ROOT


PosixPath('/Users/borisyakovlev/Documents/Jane/Current Version/investment_lab_poc')

## 1. Load the saved hypothesis
Edit the JSON in `hypotheses/` when you want a durable hypothesis.


In [4]:
HYPOTHESIS_FILE = ROOT / 'hypotheses' / 'three_up_days_then_next_day_up.json'
spec = load_hypothesis(HYPOTHESIS_FILE)
spec


{'type': 'consecutive_return_continuation_v1',
 'name': 'three_up_3pct_then_next_day_up_1pct',
 'hypothesis_text': 'If a stock rises by more than 3% close-to-close for three consecutive trading days, then on the following trading day it will rise by more than 1% close-to-close.',
 'return_basis': 'vendor_adjusted_close',
 'universe': {'symbols': ['KO', 'TSLA', 'TWTR'],
  'note': 'The free vendor sample lacks symbols.csv, so V0.3 explicitly selects the three stock tickers and excludes SPY. The full package will use instrument_type=CS instead.'},
 'signal': {'daily_return_gt': 0.03,
  'consecutive_days': 3,
  'known_after': 'close'},
 'outcome': {'horizon_trading_days': 1, 'return_gt': 0.01}}

## 2. Run it
Every run is stored under `results/<experiment_id>/` and registered in DuckDB.


In [5]:
result = run_hypothesis(spec, root=ROOT)
print(format_result(result))


Hypothesis: If a stock rises by more than 3% close-to-close for three consecutive trading days, then on the following trading day it will rise by more than 1% close-to-close.
Eligible stock-days: 334
Signals: 0
Successes: 0
Signal hit rate: n/a
Baseline next-period hit rate: 25.45%
Lift: n/a
Mean outcome return: n/a

No matching signal occurred in this dataset, so this sample cannot test the hypothesis yet.
Result JSON: results/2076762590183807912/result.json
Signal events: results/2076762590183807912/events.csv


## 3. Inspect exact output
The compiled SQL is saved alongside the result so you can audit exactly what the engine tested.


In [6]:
result_payload = json.loads((ROOT / result.result_json).read_text())
result_payload


{'experiment_id': 2076762590183807912,
 'engine_version': 'investment-lab-research-v0.3',
 'created_at': '2026-09-19T22:14:47.686404+00:00',
 'dataset_version_id': 8437122514875105285,
 'hypothesis': {'type': 'consecutive_return_continuation_v1',
  'name': 'three_up_3pct_then_next_day_up_1pct',
  'hypothesis_text': 'If a stock rises by more than 3% close-to-close for three consecutive trading days, then on the following trading day it will rise by more than 1% close-to-close.',
  'return_basis': 'vendor_adjusted_close',
  'universe': {'symbols': ['KO', 'TSLA', 'TWTR'],
   'note': 'The free vendor sample lacks symbols.csv, so V0.3 explicitly selects the three stock tickers and excludes SPY. The full package will use instrument_type=CS instead.'},
  'signal': {'daily_return_gt': 0.03,
   'consecutive_days': 3,
   'known_after': 'close'},
  'outcome': {'horizon_trading_days': 1, 'return_gt': 0.01}},
 'semantics': {'daily_return': 'close-to-close return from vendor adjusted close',
  'sign

In [7]:
print((ROOT / 'results' / str(result.experiment_id) / 'compiled.sql').read_text())



    WITH symbolized AS (
        
    SELECT
        a.*,
        coalesce(h.identifier_value, a.source_symbol) AS ticker
    FROM vendor_adjusted_daily_current a
    LEFT JOIN security_identifier_history h
      ON h.security_id = a.security_id
     AND h.source_dataset_version_id = 8437122514875105285
     AND h.identifier_type = 'TICKER'
     AND a.trade_date >= h.valid_from
     AND (h.valid_to IS NULL OR a.trade_date <= h.valid_to)
    
    ),
    filtered AS (
        SELECT * FROM symbolized
        WHERE ticker IN ('KO','TSLA','TWTR')
    ),
    daily AS (
        SELECT
            security_id,
            ticker,
            trade_date,
            adj_close,
            adj_close / lag(adj_close) OVER (
                PARTITION BY security_id ORDER BY trade_date
            ) - 1.0 AS daily_return,
            lead(adj_close, 1) OVER (
                PARTITION BY security_id ORDER BY trade_date
            ) / adj_close - 1.0 AS outcome_return
        FROM filtered
    ),

## 4. Scratchpad: change the hypothesis without editing code
For example, change 3% to 2%, or three days to two, then rerun. Save it as a JSON file only when you want to keep it.


In [ ]:
scratch = json.loads(json.dumps(spec))
scratch['signal']['daily_return_gt'] = 0.03
scratch['signal']['consecutive_days'] = 3
scratch['outcome']['return_gt'] = 0.01
scratch_result = run_hypothesis(scratch, root=ROOT)
print(format_result(scratch_result))


## Where is the code?
Open `CODE_MAP.md`. The main files are:
- `src/investment_lab/research/runner.py` — research semantics and SQL compiler
- `src/investment_lab/pipeline.py` — canonical ingestion
- `src/investment_lab/providers/historicaldata_net.py` — vendor adapter
- `schema/001_core.sql` — canonical metadata schema
- `tests/` — automated correctness checks


In [10]:
import duckdb

DB_PATH = ROOT / "warehouse" / "metadata.duckdb"
con = duckdb.connect(str(DB_PATH), read_only=True)

con.execute("DESCRIBE bars_daily_current").df()

,column_name,column_type,null,key,default,extra
0,security_id,BIGINT,YES,None,None,None
1,source_file_name,VARCHAR,YES,None,None,None
2,source_symbol,VARCHAR,YES,None,None,None
3,trade_date,DATE,YES,None,None,None
4,open,DOUBLE,YES,None,None,None
5,high,DOUBLE,YES,None,None,None
6,low,DOUBLE,YES,None,None,None
7,close,DOUBLE,YES,None,None,None
8,volume,DOUBLE,YES,None,None,None
9,vwap,DOUBLE,YES,None,None,None


In [11]:
df = load_research_frame()

NameError: name 'load_research_frame' is not defined

In [12]:
%git pull

UsageError: Line magic function `%git` not found.
